<a href="https://colab.research.google.com/github/samarreguigui/Computerlinguistik/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

from collections import Counter
import pickle
import torch
from torch.nn.utils.rnn import pad_sequence
import random

class Data:

  def __init__(self, *args):
    if len(args) == 1:
        self.init_test(args[0])
    else:
        self.init_train(args[0], args[1])
  def read_data(self, filename):
      # Colab working directory is /content
      if not os.path.isabs(filename):
          filename = os.path.join("/content", filename)

      data = []
      with open(filename, "r", encoding="utf8") as f:
          for line in f:
              line = line.strip()
              if not line:
                  continue
              parts = line.split("\t")
              if len(parts) != 2:
                  continue
              word, lemma = parts
              data.append((word, lemma))
      return data
  def make_table(self,words):
      counter = Counter()

      for w in words:
          counter.update(list(w))

      chars = [c for c, n in counter.items() if n >= 2]

      table = {}
      table["<unk>"] = 0
      table["<pad>"] = 1

      next_id = 2
      for c in sorted(chars):
          table[c] = next_id
          next_id += 1

      return table

  def init_train(self, traindata, devdata):
    self.train_data = self.read_data(traindata)
    self.dev_data = self.read_data(devdata)

    # zip für Wörter und Lemmata
    train_words, train_lemmas = zip(*self.train_data)

    self.srcChar2ID = self.make_table(train_words)
    self.numSrcChars = len(self.srcChar2ID)

    self.tgtChar2ID = self.make_table(train_lemmas)
    self.numTgtChars = len(self.tgtChar2ID)

    self.ID2tgtChar = {v: k for k, v in self.tgtChar2ID.items()}

    max_ratio = 0.0
    for word, lemma in self.train_data:
        lw = len(word)
        ll = len(lemma)
        if lw > 0:
            ratio = ll / lw
            if ratio > max_ratio:
                max_ratio = ratio

    self.max_len_factor = max_ratio
  def save(self, paramfile):
      params = {
          "srcChar2ID": self.srcChar2ID,
          "ID2tgtChar": self.ID2tgtChar,
          "max_len_factor": self.max_len_factor
      }

      with open(paramfile, "wb") as f:
          pickle.dump(params, f)

  def init_test(self, filename):
      with open(filename, "rb") as f:
          params = pickle.load(f)

      self.srcChar2ID = params["srcChar2ID"]
      self.ID2tgtChar = params["ID2tgtChar"]
      self.max_len_factor = params["max_len_factor"]

  def batches(self, data, max_batch_size):
      batch_words = []
      batch_lemmas = []
      current_size = 0

      def process_batch(words, lemmas):
          # Eingabesequenzen
          src_seqs = []
          src_lengths = []

          for w in words:
              ids = [self.srcChar2ID.get(c, self.srcChar2ID["<unk>"]) for c in w]
              src_lengths.append(len(ids))
              src_seqs.append(torch.tensor(ids, dtype=torch.long))

          srcIDvecs = pad_sequence(
              src_seqs,
              batch_first=False,
              padding_value=self.srcChar2ID["<pad>"]
          )

          # Ausgabesequenzen
          tgt_seqs = []
          for l in lemmas:
              ids = [self.tgtChar2ID.get(c, self.tgtChar2ID["<unk>"]) for c in l]
              ids = [self.tgtChar2ID["<pad>"]] + ids + [self.tgtChar2ID["<pad>"]]
              tgt_seqs.append(torch.tensor(ids, dtype=torch.long))

          tgtIDvecs = pad_sequence(
              tgt_seqs,
              batch_first=False,
              padding_value=self.tgtChar2ID["<pad>"]
          )

          return srcIDvecs, src_lengths, tgtIDvecs

      for word, lemma in data:
          cost = len(word) + len(lemma)

          if batch_words and current_size + cost > max_batch_size:
              yield process_batch(batch_words, batch_lemmas)
              batch_words = []
              batch_lemmas = []
              current_size = 0

          batch_words.append(word)
          batch_lemmas.append(lemma)
          current_size += cost

      if batch_words:
          yield process_batch(batch_words, batch_lemmas)


  def train_batches(self, max_batch_size):
        random.shuffle(self.train_data)
        return self.batches(self.train_data, max_batch_size)



  def dev_batches(self, max_batch_size):
      return self.batches(self.dev_data, max_batch_size)



  def test_batches(self, file, max_batch_size):
      # Wörter aus Datei lesen
      words = []
      with open(file, "r", encoding="utf8") as f:
          for line in f:
              w = line.strip()
              if w:
                  words.append(w)

      batch_words = []
      current_size = 0

      def process_batch(srcs):
          srcID = []
          src_lengths = []

          for w in srcs:
              ids = [self.srcChar2ID.get(c, self.srcChar2ID["<unk>"]) for c in w]
              src_lengths.append(len(ids))
              srcID.append(torch.tensor(ids, dtype=torch.long))

          srcIDvecs = pad_sequence(
              srcID,
              batch_first=False,
              padding_value=self.srcChar2ID["<pad>"]
          )

          maxTgtLen = int(max(src_lengths) * self.max_len_factor + 4)

          return srcs, srcIDvecs, src_lengths, maxTgtLen

      for w in words:
          cost = len(w)

          if batch_words and current_size + cost > max_batch_size:
              yield process_batch(batch_words)
              batch_words = []
              current_size = 0

          batch_words.append(w)
          current_size += cost

      if batch_words:
          yield process_batch(batch_words)

  def tgtIDs2chars(self, tgtCharIDs):
      pad_id = self.tgtChar2ID["<pad>"]
      chars = []

      for idx in tgtCharIDs:
          if idx == pad_id:
              break
          chars.append(self.ID2tgtChar.get(idx, "<unk>"))

      return chars


if __name__ == "__main__":
    # Data Objekt erzeugen und Trainingsmodus verwenden
    data = Data("train.txt", "dev.txt")

    print("Anzahl Source Characters:", data.numSrcChars)
    print("Anzahl Target Characters:", data.numTgtChars)
    print("Max Length Factor:", data.max_len_factor)

    # Erstes Batch aus den Trainingsdaten prüfen
    for srcIDvecs, src_lengths, tgtIDvecs in data.train_batches(200):
        print("srcIDvecs Größe:", srcIDvecs.size())
        print("tgtIDvecs Größe:", tgtIDvecs.size())
        print("src_lengths:", src_lengths)
        break

    # Speichern
    data.save("params.pkl")

    # Laden im Testmodus
    test_data = Data("params.pkl")
    print("Geladen im Testmodus. Max Length Factor:", test_data.max_len_factor)


Anzahl Source Characters: 101
Anzahl Target Characters: 92
Max Length Factor: 0.8260869565217391
srcIDvecs Größe: torch.Size([35, 4])
tgtIDvecs Größe: torch.Size([23, 4])
src_lengths: [20, 35, 33, 25]
Geladen im Testmodus. Max Length Factor: 0.8260869565217391


In [4]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_size, lstm_size, dropout_rate):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_size)

        # 3-layer bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=embedding_size,
            hidden_size=lstm_size,
            num_layers=3,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_rate
        )

        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, src_letter_ids, src_lengths):
        embs = self.dropout(self.embedding(src_letter_ids))

        # Pack the padded sequence
        packed = pack_padded_sequence(
            embs,
            src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # Run the BiLSTM
        packed_output, (hidden, cell) = self.lstm(packed)

        # Unpack the output
        output, _ = pad_packed_sequence(packed_output, batch_first=True)

        # Extract LAST LAYER backward hidden state: hidden[-1]
        backward_hidden_last = hidden[-1:].contiguous()
        backward_cell_last = cell[-1:].contiguous()

        return output, backward_hidden_last, backward_cell_last

class Attention(nn.Module):
    def __init__(self, lstm_size, dropout_rate):
        super().__init__()

        self.ff1 = nn.Linear(3 * lstm_size, lstm_size)
        self.ff2 = nn.Linear(lstm_size, 1)

        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, encoder_states, decoder_states):

        batch_size, enc_len, _ = encoder_states.size()
        _, dec_len, _ = decoder_states.size()

        # Expand decoder states to match encoder length
        dec_exp = decoder_states.unsqueeze(2).expand(batch_size, dec_len, enc_len, -1)

        # Expand encoder states to match decoder steps
        enc_exp = encoder_states.unsqueeze(1).expand(batch_size, dec_len, enc_len, -1)

        # Concatenate encoder+decoder states
        combined = torch.cat([enc_exp, dec_exp], dim=-1)

        # Compute attention scores
        scores = self.ff2(torch.tanh(self.ff1(self.dropout(combined)))).squeeze(-1)

        # Softmax over encoder positions
        attn = torch.softmax(scores, dim=2).unsqueeze(-1)

        context = torch.sum(attn * enc_exp, dim=2)

        return context

class Model(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, embedding_size, lstm_size, dropout_rate, pad_id):
        super().__init__()

        self.pad_id = pad_id

        # Encoder + Attention
        self.encoder = Encoder(src_vocab_size, embedding_size, lstm_size, dropout_rate)
        self.attention = Attention(lstm_size, dropout_rate)

        # Decoder Embedding
        self.embedding = nn.Embedding(tgt_vocab_size, embedding_size)

        # Three unidirectional LSTMs for the decoder
        self.lstm1 = nn.LSTM(embedding_size, lstm_size, batch_first=True)
        self.lstm2 = nn.LSTM(lstm_size + 2*lstm_size, lstm_size, batch_first=True)
        self.lstm3 = nn.LSTM(lstm_size + 2*lstm_size, lstm_size, batch_first=True)

        # Projection layer
        self.output_projection = nn.Linear(lstm_size, embedding_size)

        # Output layer: embedding → vocabulary logits
        self.output_layer = nn.Linear(embedding_size, tgt_vocab_size, bias=False)

        # Tied embeddings: input and output share the same weight matrix
        self.output_layer.weight = self.embedding.weight

        # Dropout
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, src_letter_ids, src_lengths, tgt_letter_ids):

        enc_states, enc_hidden, enc_cell = self.encoder(src_letter_ids, src_lengths)

        batch_size = src_letter_ids.size(0)

        #Embed target characters
        embs = self.dropout(self.embedding(tgt_letter_ids))

        # LSTM1: initializes with encoder backward states
        dec_states1, states1 = self.lstm1(embs, (enc_hidden, enc_cell))

        #Attention for LSTM1 output
        context1 = self.attention(enc_states, dec_states1)

        lstm2_input = torch.cat([dec_states1, context1], dim=-1)

        # LSTM2
        dec_states2, states2 = self.lstm2(lstm2_input, (enc_hidden, enc_cell))

        # Attention for LSTM2 output
        context2 = self.attention(enc_states, dec_states2)
        lstm3_input = torch.cat([dec_states2, context2], dim=-1)

        # LSTM3
        dec_states3, states3 = self.lstm3(lstm3_input, (enc_hidden, enc_cell))

        # Projection to embedding size
        proj = self.output_projection(self.dropout(dec_states3))

        #Output layer to vocabulary logits
        logits = self.output_layer(proj)

        return logits

    def lemmatize(self, src_letter_ids, src_lengths, max_tgt_length):
        self.eval()

        #Encode
        enc_states, enc_hidden, enc_cell = self.encoder(src_letter_ids, src_lengths)

        batch_size = src_letter_ids.size(0)
        pad_id = self.pad_id
        # Start with PAD token for every batch element
        tgt_char_ids = torch.full((batch_size, 1), pad_id, dtype=torch.long, device=src_letter_ids.device)

        # Initialize previous states
        prev_states1 = (enc_hidden, enc_cell)
        prev_states2 = (enc_hidden, enc_cell)
        prev_states3 = (enc_hidden, enc_cell)

        all_tgt_char_ids = []

        for _ in range(max_tgt_length):

            #Embedding
            embs = self.dropout(self.embedding(tgt_char_ids))

            # LSTM1
            dec_states1, prev_states1 = self.lstm1(embs, prev_states1)

            # Attention
            context1 = self.attention(enc_states, dec_states1)

            # LSTM2
            lstm2_input = torch.cat([dec_states1, context1], dim=-1)
            dec_states2, prev_states2 = self.lstm2(lstm2_input, prev_states2)

            # Attention again
            context2 = self.attention(enc_states, dec_states2)

            # LSTM3
            lstm3_input = torch.cat([dec_states2, context2], dim=-1)
            dec_states3, prev_states3 = self.lstm3(lstm3_input, prev_states3)

            # Output
            proj = self.output_projection(dec_states3)
            logits = self.output_layer(proj)

            next_char = logits.argmax(dim=-1)[:, -1]  # last timestep
            all_tgt_char_ids.append(next_char)

            # Prepare next input
            tgt_char_ids = next_char.unsqueeze(1)

            # Stop if all sequences have at least one PAD symbol
            all_generated = torch.stack(all_tgt_char_ids, dim=1)
            if torch.all(torch.any(all_generated == pad_id, dim=1), dim=0):
                break

        return torch.stack(all_tgt_char_ids, dim=1)

    @torch.no_grad()
    def predict(self, src_letter_ids, src_lengths, max_tgt_length):
        # Encode input
        enc_states, enc_hidden, enc_cell = self.encoder(src_letter_ids, src_lengths)
        batch_size = src_letter_ids.size(0)

        #Prepare output storage
        all_tgt_char_ids = []

        # Start with PAD token for every batch element
        tgt_char_ids = torch.full((batch_size, 1), self.pad_id, dtype=torch.long, device=src_letter_ids.device)

        # Initial states
        states1 = (enc_hidden, enc_cell)
        states2 = (enc_hidden, enc_cell)
        states3 = (enc_hidden, enc_cell)

        for t in range(max_tgt_length):

            #Embed last generated character
            embs = self.embedding(tgt_char_ids)  # (batch, 1, emb_size)

            # --- LSTM1 ---
            dec_states1, states1 = self.lstm1(embs, states1)

            # Attention → context
            context1 = self.attention(enc_states, dec_states1)

            # concat context with LSTM1 output
            in2 = torch.cat([dec_states1, context1], dim=-1)

            # --- LSTM2 ---
            dec_states2, states2 = self.lstm2(in2, states2)

            context2 = self.attention(enc_states, dec_states2)
            in3 = torch.cat([dec_states2, context2], dim=-1)

            # --- LSTM3 ---
            dec_states3, states3 = self.lstm3(in3, states3)

            # Projection + output
            proj = self.output_projection(dec_states3)
            logits = self.output_layer(proj)

            # Greeedy decoding: pick the argmax
            next_char = torch.argmax(logits, dim=-1)
            tgt_char_ids = next_char

            # Save generated character
            all_tgt_char_ids.append(next_char.squeeze(1))

            # Stopp if ALL sequences output PAD in this step
            all_generated = torch.stack(all_tgt_char_ids, dim=1)
            if torch.all(torch.any(all_generated == self.pad_id, dim=1), dim=0):
                break

        # Final output: shape (batch, decoded_length)
        return torch.stack(all_tgt_char_ids, dim=1)

In [ ]:
import argparse
import torch
import torch.nn as nn
from torch.optim import AdamW

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
import sys

sys.argv = [
    "lemmatizer-train.py",
    "train.txt",
    "dev.txt",
    "model"
]

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("trainfile")
    parser.add_argument("devfile")
    parser.add_argument("paramfile")

    parser.add_argument("--embeddings_size", type=int, default=100)
    parser.add_argument("--lstm_size", type=int, default=400)
    parser.add_argument("--num_epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=3000)
    parser.add_argument("--dropout_rate", type=float, default=0.5)

    args = parser.parse_args()

    # Data
    data = Data(args.trainfile, args.devfile)
    data.save(args.paramfile + ".io")

    num_src_chars = len(data.srcChar2ID)
    num_tgt_chars = len(data.ID2tgtChar)

    # Model
    model = Model(
        num_src_chars,
        num_tgt_chars,
        args.embeddings_size,
        args.lstm_size,
        args.dropout_rate,
        1
    ).to(DEVICE)

    loss_fn = nn.CrossEntropyLoss(ignore_index=1)
    optimizer = AdamW(model.parameters())

    best_acc = 0.0

    for epoch in range(1, args.num_epochs + 1):

        # -------- TRAIN --------
        model.train()
        for src_ids, src_lengths, tgt_ids in data.train_batches(args.batch_size):

            # FIX: transpose from (seq_len, batch) → (batch, seq_len)
            src_ids = src_ids.transpose(0, 1).to(DEVICE)
            tgt_ids = tgt_ids.transpose(0, 1).to(DEVICE)

            optimizer.zero_grad()

            src_lengths = torch.tensor(src_lengths, device=src_ids.device)

            logits = model(src_ids, src_lengths, tgt_ids[:, :-1])

            loss = loss_fn(
                logits.reshape(-1, logits.size(-1)),
                tgt_ids[:, 1:].reshape(-1)
            )

            loss.backward()
            optimizer.step()

        # -------- EVAL --------
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for src_ids, src_lengths, tgt_ids in data.dev_batches(args.batch_size):

                src_ids = src_ids.transpose(0, 1).to(DEVICE)
                tgt_ids = tgt_ids.transpose(0, 1).to(DEVICE)

                src_lengths = torch.tensor(src_lengths, device=src_ids.device)

                logits = model(
                    src_ids,
                    src_lengths,
                    tgt_ids[:, :-1]
)

                preds = logits.argmax(dim=-1)
                mask = tgt_ids[:, 1:] != 1

                correct += ((preds == tgt_ids[:, 1:]) & mask).sum().item()
                total += mask.sum().item()

        acc = correct / total
        print(acc)

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), args.paramfile + ".pth")

if __name__ == "__main__":
    main()


0.9959140635297219
0.9966484023423525
0.9978346419627558
0.9982488843698808
0.9982488843698808
0.9983995179724717
0.998493663974091
0.998493663974091
0.9983053719708525
0.9982488843698808
0.9988702479805682
0.9987384435783012
0.9988890771808921
